# Module 01: Information Theory & Partial Information Decomposition (PID)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#0F172A'
plt.rcParams['axes.linewidth'] = 1.2

def build_joint_distribution(gate_type='XOR', input_corr=0.0, noise_prob=0.0):
    p_joint = np.zeros((2, 2, 2), dtype=np.float64)
    for x1 in (0, 1):
        for x2 in (0, 1):
            sign = 1.0 if x1 == x2 else -1.0
            p_x1_x2 = 0.25 * (1.0 + sign * input_corr)
            if gate_type == 'XOR':
                y_det = x1 ^ x2
            elif gate_type == 'AND':
                y_det = x1 & x2
            elif gate_type == 'OR':
                y_det = x1 | x2
            elif gate_type == 'COPY_X1':
                y_det = x1
            else:
                y_det = x1 ^ x2
            for y in (0, 1):
                p_y_given_x = (1.0 - noise_prob) if y == y_det else noise_prob
                p_joint[x1, x2, y] = p_x1_x2 * p_y_given_x
    p_joint /= np.sum(p_joint)
    return p_joint

def compute_shannon_entropy(p):
    p_flat = p.flatten()
    p_pos = p_flat[p_flat > 1e-15]
    return -np.sum(p_pos * np.log2(p_pos))

def compute_specific_information(p_joint_xy, y_val):
    p_x = np.sum(p_joint_xy, axis=-1)
    p_y = np.sum(p_joint_xy, axis=tuple(range(p_joint_xy.ndim - 1)))
    if p_y[y_val] <= 1e-15:
        return 0.0
    p_x_given_y = p_joint_xy[..., y_val] / p_y[y_val]
    mask = (p_x_given_y > 1e-15) & (p_x > 1e-15)
    return float(np.sum(p_x_given_y[mask] * np.log2(p_x_given_y[mask] / p_x[mask])))

def compute_pid_williams_beer(p_joint):
    p_x1_y = np.sum(p_joint, axis=1)
    p_x2_y = np.sum(p_joint, axis=0)
    p_y = np.sum(p_joint, axis=(0, 1))
    p_x1_x2 = np.sum(p_joint, axis=2)
    h_y = compute_shannon_entropy(p_y)
    h_x1_x2 = compute_shannon_entropy(p_x1_x2)
    h_joint = compute_shannon_entropy(p_joint)
    mi_joint = h_x1_x2 + h_y - h_joint
    mi_x1_y = compute_shannon_entropy(np.sum(p_x1_y, axis=1)) + h_y - compute_shannon_entropy(p_x1_y)
    mi_x2_y = compute_shannon_entropy(np.sum(p_x2_y, axis=1)) + h_y - compute_shannon_entropy(p_x2_y)
    redundancy = 0.0
    for y_val in range(p_y.shape[0]):
        if p_y[y_val] > 1e-15:
            i_spec_1 = compute_specific_information(p_x1_y, y_val)
            i_spec_2 = compute_specific_information(p_x2_y, y_val)
            redundancy += p_y[y_val] * min(i_spec_1, i_spec_2)
    unique_x1 = max(0.0, mi_x1_y - redundancy)
    unique_x2 = max(0.0, mi_x2_y - redundancy)
    synergy = max(0.0, mi_joint - (redundancy + unique_x1 + unique_x2))
    return {'H(Y)': h_y, 'I(X1; Y)': mi_x1_y, 'I(X2; Y)': mi_x2_y, 'I(X1, X2; Y)': mi_joint, 'Redundancy': redundancy, 'Unique_X1': unique_x1, 'Unique_X2': unique_x2, 'Synergy': synergy}

def run_pid_dashboard(gate_type='XOR', input_corr=0.0, noise_prob=0.0):
    p_joint = build_joint_distribution(gate_type, input_corr, noise_prob)
    res = compute_pid_williams_beer(p_joint)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    fig.patch.set_facecolor('#F8FAFC')
    ax1 = axes[0]
    ax1.set_facecolor('#F8FAFC')
    mi_labels = ['I(X1; Y)', 'I(X2; Y)', 'I(X1, X2; Y)']
    mi_vals = [res['I(X1; Y)'], res['I(X2; Y)'], res['I(X1, X2; Y)']]
    bars1 = ax1.bar(mi_labels, mi_vals, color=['#64748B', '#64748B', '#2563EB'], width=0.52, edgecolor='#0F172A', linewidth=1.2)
    ax1.set_ylim(0, 1.15)
    ax1.set_ylabel('Information (Bits)', fontsize=11, fontweight='bold')
    ax1.set_title('1. Pairwise vs. Joint MI (' + gate_type + ')', fontsize=12, fontweight='bold')
    ax1.grid(axis='y', linestyle='--', alpha=0.4)
    for bar in bars1:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2, h + 0.03, f'{h:.3f} b', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax2 = axes[1]
    ax2.set_facecolor('#F8FAFC')
    pid_labels = ['Redundancy', 'Unique X1', 'Unique X2', 'Synergy']
    pid_vals = [res['Redundancy'], res['Unique_X1'], res['Unique_X2'], res['Synergy']]
    bars2 = ax2.bar(pid_labels, pid_vals, color=['#0D9488', '#2563EB', '#D97706', '#E11D48'], width=0.55, edgecolor='#0F172A', linewidth=1.2)
    ax2.set_ylim(0, 1.15)
    ax2.set_ylabel('Decomposed Information (Bits)', fontsize=11, fontweight='bold')
    ax2.set_title('2. Williams & Beer PID Decomposition', fontsize=12, fontweight='bold')
    ax2.grid(axis='y', linestyle='--', alpha=0.4)
    for bar in bars2:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.03, f'{h:.3f} b', ha='center', va='bottom', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f"H(Y)={res['H(Y)']:.4f} b | Joint MI={res['I(X1, X2; Y)']:.4f} b | Synergy={res['Synergy']:.4f} b")

widgets.interact(
    run_pid_dashboard,
    gate_type=widgets.Dropdown(options=['XOR', 'AND', 'OR', 'COPY_X1'], value='XOR', description='Mechanism:'),
    input_corr=widgets.FloatSlider(min=-0.9, max=0.9, step=0.05, value=0.0, description='Input Corr:'),
    noise_prob=widgets.FloatSlider(min=0.0, max=0.45, step=0.05, value=0.0, description='Channel Noise:'),
);
